## PageIndex - Vectorless RAG 

Reasoning-based RAG with No Vector DB, No Chunking 

In [2]:
import os, json, time 
from dotenv import load_dotenv

load_dotenv()

PAGEINDEX_API_KEY = os.getenv('PAGEINDEX_API_KEY')
GROQ_API_KEY = os.getenv('GROQ_API_KEY')

print("PageIndex key loaded:", "✅" if PAGEINDEX_API_KEY else "❌ Missing!")
print("GroqApi key loaded:   ", "✅" if GROQ_API_KEY    else "❌ Missing!")

PageIndex key loaded: ✅
GroqApi key loaded:    ✅


In [4]:
from pageindex import PageIndexClient
from openai import OpenAI

pi_client = PageIndexClient(api_key=PAGEINDEX_API_KEY)

# Point the OpenAI client at Groq's OpenAI-compatible endpoint
groq_client = OpenAI(
    api_key=os.getenv("GROQ_API_KEY"),
    base_url="https://api.groq.com/openai/v1"
)

print("✅ PageIndex client ready")
print("✅ Groq client ready")

✅ PageIndex client ready
✅ Groq client ready


### Upload and Index a PDF 

What happens here: 
 - upload your PDF to the PageIndex cloud 
 - PageIndex uses an LLM to read the document structure 
 - Builds a hierarchical tree index (like a smart Table of Contents)
 - Returns a doc_id for all future operations 


 Why no chunking ? 
 Instead of cutting the document into arbitary 500 token pieces, PageIndex respects the document's natural section boundaries -- chapters, sub-sections, paragraphs --- as the author intended.

In [7]:
# upload your pdf 
# replace with the path to you pdf file 
# Great candidates: Annual report, research papers, legal docs, textbooks 

PDF_PATH = './data/pdf/attention.pdf'

print(f"Uploading: {PDF_PATH}")
result = pi_client.submit_document(PDF_PATH)
doc_id = result["doc_id"]

print(f"✅ Uploaded!")
print(f"📋 Document ID: {doc_id}")
print("   (Save this ID — you'll use it throughout the notebook)")

Uploading: ./data/pdf/attention.pdf
✅ Uploaded!
📋 Document ID: pi-cmt48dp8x01s301p5izu33rwl
   (Save this ID — you'll use it throughout the notebook)


In [8]:
#poll until process is complete 
# Page index builds tree asynchronously 
# for a 50 page PDF this typically takes 30-90 seconds 

print("building tree index")
print("    (This runs once per document - the index is cached for reuse)")

while True: 
    status_result = pi_client.get_document(doc_id)
    status = status_result.get("status")
    print(f"   Status: {status}")

    if status == "completed": 
        print("\n Tree index ready")
        break
    elif status == "falied": 
        print("\n Processing failed. Check your PDF format")
        break

    time.sleep(5)

building tree index
    (This runs once per document - the index is cached for reuse)
   Status: completed

 Tree index ready


### Inspect the Tree Structure 

What the tree looks like: 

    Document
    ├── Introduction (pages 1-3)
    │   └── Background (pages 1-2)
    ├── Financial Stability (pages 21-31)
    │   ├── Monitoring Vulnerabilities (pages 22-28)
    │   └── International Cooperation (pages 28-31)
    └── Conclusion (pages 45-47)

Each node has: 
- node_id : unique ID used during retrieval 
- title : section heading 
- page_index : page number in original PDF 
- text : section summary (when node_summary = True)
- nodes : child sections (nested)

This structure is what the LLM reasons over during retrieval 

In [9]:
# Fetch the full tree 
tree_result = pi_client.get_tree(doc_id, node_summary = True)
pageindex_tree = tree_result.get("result", [])

print(f"📊 Top-level sections: {len(pageindex_tree)}")
print("\n🌲 Raw tree (first node):")
print(json.dumps(pageindex_tree[0] if pageindex_tree else {}, indent=2))

📊 Top-level sections: 1

🌲 Raw tree (first node):
{
  "title": "Attention Is All You Need",
  "node_id": "0000",
  "page_index": 1,
  "prefix_summary": "# Attention Is All You Need\n\n**Ashish Vaswani***\nGoogle Brain\navaswani@google.com\n\n**Noam Shazeer***\nGoogle Brain\nnoam@google.com\n\n**Niki Parmar***\nGoogle Research\nnikip@google.com\n\n**Jakob Uszkoreit***\nGoogle Research\nusz@google.com\n\n**Llion Jones***\nGoogle Research\nllion@google.com\n\n**Aidan N. Gomez*** \u2020\nUniversity of Toronto\naidan@cs.toronto.edu\n\n**\u0141ukasz Kaiser***\nGoogle Brain\nlukaszkaiser@google.com\n\n**Illia Polosukhin*** \u2021\nillia.polosukhin@gmail.com\n",
  "text": "# Attention Is All You Need\n\n**Ashish Vaswani***\nGoogle Brain\navaswani@google.com\n\n**Noam Shazeer***\nGoogle Brain\nnoam@google.com\n\n**Niki Parmar***\nGoogle Research\nnikip@google.com\n\n**Jakob Uszkoreit***\nGoogle Research\nusz@google.com\n\n**Llion Jones***\nGoogle Research\nllion@google.com\n\n**Aidan N. Gomez**

In [10]:
# Pretty print the full tree 
def print_tree(nodes, indent=0):
    """Recursively print tree titles for a visual overview."""
    for node in nodes:
        prefix = "  " * indent + ("└─ " if indent > 0 else "")
        page   = node.get("page_index", "?")
        print(f"{prefix}[{node['node_id']}] {node['title']}  (p.{page})")
        if node.get("nodes"):
            print_tree(node["nodes"], indent + 1)

print("📚 Full Document Structure:\n")
print_tree(pageindex_tree)

📚 Full Document Structure:

[0000] Attention Is All You Need  (p.1)
  └─ [0001] Abstract  (p.1)
  └─ [0002] 1 Introduction  (p.2)
  └─ [0003] 2 Background  (p.2)
  └─ [0004] 3 Model Architecture  (p.2)
    └─ [0005] 3.1 Encoder and Decoder Stacks  (p.3)
    └─ [0006] 3.2 Attention  (p.3)
      └─ [0007] 3.2.1 Scaled Dot-Product Attention  (p.4)
      └─ [0008] 3.2.2 Multi-Head Attention  (p.4)
      └─ [0009] 3.2.3 Applications of Attention in our Model  (p.5)
    └─ [0010] 3.3 Position-wise Feed-Forward Networks  (p.5)
    └─ [0011] 3.4 Embeddings and Softmax  (p.5)
    └─ [0012] 3.5 Positional Encoding  (p.6)
  └─ [0013] 4 Why Self-Attention  (p.6)
  └─ [0014] 5 Training  (p.7)
  └─ [0015] 6 Results  (p.8)
    └─ [0016] 6.1 Machine Translation  (p.8)
    └─ [0017] 6.2 Model Variations  (p.8)
    └─ [0018] 6.3 English Constituency Parsing  (p.9)
  └─ [0019] 7 Conclusion  (p.10)
  └─ [0020] References  (p.10)
  └─ [0021] Attention Visualizations  (p.13)


In [11]:
# count total nodes 
def count_nodes(nodes): 
    total = len(nodes)
    for n in nodes: 
        if n.get("nodes"): 
            total += count_nodes(n['nodes'])
    return total 

total= count_nodes(pageindex_tree)
print(f"🔢 Total nodes in tree: {total}")
print("   Each node = one retrievable section of the document")

🔢 Total nodes in tree: 22
   Each node = one retrievable section of the document


### LLM Tree Search -- The Core of PageIndex 

This is where PageIndex fundamentally differs from vector RAG 

#### Vector RAG retreival: 
    query → embed → cosine_similarity(query_vec, all_chunk_vecs) → top-k chunks

*Problem: finds what's similar, not what's relevant*


#### PageIndex retreival: 
    query + tree → LLM reasons → "node 0007 and 0008 contain the answer"

*Advantage: LLM understands document structure, context, and intent*


The LLM acts like a human expert scanning a Table of Contents


In [67]:
## LLM Tree Search Function 

def llm_tree_search(query: str, tree: list, model: str ="openai/gpt-oss-20b") -> dict: 
    """
    Core PageIndex retreival: 
    Sends the query + document tree to an LLM 
    LLM reasons over the structure and returns relevant node_ids. 

    Returns: dict with 'thinking' (reasoning) and 'node_list' (node IDs)
    
    """

    # Compress tree to save tokens - only send titles + short summaries 
    def compress(nodes):
        out = []
        for n in nodes:
            entry = {
                "node_id": n["node_id"],
                "title":   n["title"],
                "page":    n.get("page_index", "?"),
                "summary": n.get("text", "")[:150]  # first 150 chars
            }
            if n.get("nodes"):
                entry["children"] = compress(n["nodes"])
            out.append(entry)
        return out
    
    compressed_tree = compress(tree)

    prompt = f"""You are given a query and a document's tree structure (like a Table of Content). 
    Your task: identify which node IDs most likely contain the answer to the query. 
    Think step-by-step about which sections are relevant

    Query: {query}

    Document Tree: 
    {json.dumps(compressed_tree, indent = 2)}

    Reply ONLY in this exact JSON format: 
    {{
       "thinking": "<your step-by-step reasoning>",
        "node_list": ["node_id1", "node_id2"]
    }}
    """

    response = groq_client.chat.completions.create(
        model=model,
        messages=[{"role": "user", "content": prompt}], 
        response_format={"type": "json_object"}
    )

    return json.loads(response.choices[0].message.content)



In [68]:
# Test with sample query 
query = "What is the syllabus covered in attention is all you need?"

print(f"🔍 Query: {query}\n")
result = llm_tree_search(query, pageindex_tree)

print("🧠 LLM Reasoning:")
print(result.get("thinking", "N/A"))
print()
print("🎯 Selected Node IDs:", result.get("node_list", []))

🔍 Query: What is the syllabus covered in attention is all you need?

🧠 LLM Reasoning:
The query asks for the syllabus of the paper "Attention Is All You Need," which is essentially the table of contents. The main sections of the paper are represented by top‑level nodes in the provided document tree. These include Abstract (0001), Introduction (0002), Background (0003), Model Architecture (0004), Why Self‑Attention (0013), Training (0014), Results (0015), and Conclusion (0019). These nodes collectively cover the entire syllabus of the paper.

🎯 Selected Node IDs: ['0001', '0002', '0003', '0004', '0013', '0014', '0015', '0019']


### Full End-to-End RAG Pipeline

3 steps: 
- Tree Search : LLM picks relevant node_ids 
- Retrieve : Fetch the actual section cotent from those nodes 
- Generate : LLM writes a grounded answer with page citations 

What makes this better than vector RAG: 
- Retreived content has titles + page numbers (traceable)
- LLM can cite exactly which section the answer comes from 
- No hallicination from irrelevant chunks 

In [69]:
# Helper function: find nodes by ID 
def find_nodes_by_ids(tree: list, target_ids: list)-> list: 
    """Recursively walk the tree and collect nodes matching target ids"""
    found = []
    for node in tree: 
        if node['node_id'] in target_ids: 
            found.append(node)
        if node.get("nodes"): 
            found.extend(find_nodes_by_ids(node["nodes"], target_ids))
    return found 

In [70]:
# generate answer from retreived node 
def generate_answer(query: str, nodes: list, model: str = "openai/gpt-oss-20b") -> str: 
    """
    Takes retrieved node  as context and generates a grounded answer. 
    Instructs the LLM to cite section titles and page numbers. 
    """
    if not nodes: 
        return "no relevant sections found in the document"

    # Buidl a context string form retrieved nodes 
    context_parts = []
    for node in nodes: 
        context_parts.append(
            f"[Section: '{node['title']}' | Page {node.get('page_index', '?')}]\n"\
            f"{node.get('text', 'Content not available')}"
        )
    context = "\n\n---\n\n".join(context_parts)

    prompt = f"""You are an expert document analyst. Answer the quesiton using ONLY the provided context. For every claim you make, cite the section title and page number in parentheses. 
    Be concise and precise.
    
    Question: {query}
    
    Context: 
    {context}
    
    Answer:"""

    response = groq_client.chat.completions.create(
        model=model,
        messages=[
            {"role": "user", "content": prompt}
        ]
    )

    return response.choices[0].message.content

In [71]:
# The complete vectorless RAG function 

def vectorless_rag(query: str, tree: list, verbose: bool = True) -> str: 
    """
    Full end-to-end PageIndex pipeline: 

    Step 1: LLM Tree Search : finds relevant node_ids 
    Step 2: Node Retreival : fetches section content 
    Step 3: Answer Generation : produces cited answer 
    """

    if verbose:
        print(f"{'='*55}")
        print(f"🔍 Query: {query}")
        print(f"{'='*55}")


    # tree search 
    search_result = llm_tree_search(query, tree) 
    node_ids = search_result.get("node_list", [])

    if verbose:
        print(f"\n🧠 Reasoning: {search_result.get('thinking', '')[:200]}...")
        print(f"🎯 Retrieved node IDs: {node_ids}")

    # Step 2: Retrieve nodes
    nodes = find_nodes_by_ids(tree, node_ids)
    
    if verbose:
        print(f"📄 Sections found: {[n['title'] for n in nodes]}")
    
    # Step 3: Generate answer
    answer = generate_answer(query, nodes)
    
    if verbose:
        print(f"\n📝 Answer:\n{answer}")
    
    return answer

In [72]:
# Run the full pipeline 
answer = vectorless_rag(
    query="What are the syllabus covered in attnetion is all you need?",
    tree=pageindex_tree
)



🔍 Query: What are the syllabus covered in attnetion is all you need?

🧠 Reasoning: The question asks for the syllabus covered in the paper "Attention Is All You Need." The syllabus corresponds to the main sections of the paper: Abstract, Introduction, Background, Model Architecture,...
🎯 Retrieved node IDs: ['0001', '0002', '0003', '0004', '0013', '0014', '0015', '0019']
📄 Sections found: ['Abstract', '1 Introduction', '2 Background', '3 Model Architecture', '4 Why Self-Attention', '5 Training', '6 Results', '7 Conclusion']

📝 Answer:
The paper’s outline (its “syllabus”) includes the following sections:

- **Abstract** (Page 1)  
- **1 Introduction** (Page 2)  
- **2 Background** (Page 2)  
- **3 Model Architecture** (Page 2)  
- **4 Why Self‑Attention** (Page 6)  
- **5 Training** (Page 7)  
- **6 Results** (Page 8)  
- **7 Conclusion** (Page 10)


### Expert-Guided Retrieval 

The killer feature no one talks about. 

With vector RAG, injecting domain expertise requires fine-tuning the embedding model - expensive and time consumin g

With PageIndex, you just add rules to the prompt. 

        "If the query mentions EBITDA → prioritize the MD&A section"
        "If the query is about risks  → check Part I, Item 1A"

This makes PageIndex instantly adaptable to any domain - finance, legal, medical, technical - without any model training 

### Analogy to understand 


Okay, imagine this like a **treasure hunt in a giant library**. 🏰📚

## The basic version (no expert)

You give a robot helper a question — like "where do dinosaurs eat?" — and a map of the library (the table of contents). The robot reads through the whole map, room by room, guessing which room might have the answer. It's smart, but it's doing all the guessing from scratch every single time, like a new kid on their first day at the library.

## The expert-guided version

Now imagine the library has an old, wise librarian who's worked there for 30 years. She doesn't need to guess — she just *knows*:

> "Dinosaur questions? Room 4, third shelf.
> Space questions? Basement, next to the telescopes.
> Ocean questions? Room 2, behind the big fish poster."

So before you send the robot in, you hand it a little cheat-sheet from the librarian — a list of "if it's about *this*, go look *there*." That cheat-sheet is exactly what `FINANCIAL_EXPERT_RULES` is in the notebook. It's just a note, written in plain words, like:

```
EBITDA questions      → go to the MD&A shelf
Risk questions        → go to the Risk Factors shelf
Finetuning questions  → go to Module 9
```

Now when the robot goes hunting, it's not just wandering and guessing — it's wandering **with a cheat-sheet in its pocket**, so it finds the right room much faster and more reliably.

## The cool part

To teach a *robot dog* (the old vector-search way) a new trick, you'd have to retrain it for weeks — running it through the whole yard over and over until it learns. Expensive, slow.

But this robot helper doesn't need retraining at all. You just **hand it a new cheat-sheet**, and it instantly knows the new rules — because it can *read*. Want it to be a doctor's assistant instead of a librarian's assistant? Just write a new cheat-sheet: "fever questions → go here, broken bone questions → go there." No retraining, no weeks of practice. Just swap the note.

That's the whole trick: **`llm_tree_search_with_expert()` is the exact same robot as before — same map, same job — except now it also gets to read the wise librarian's cheat-sheet before it starts hunting.** Everything else (finding the actual books, and writing you an answer with the page number) stays exactly the same.

In [73]:
# Define domain expert rules 
# These are routing rules that tell the LLM WHERE to look for specific queries. 
# Think of it as encoding a senior analyst's institutional knowledge 

In [74]:
FINANCIAL_EXPERT_RULES = """
Expert routing rules for financial documents (10-K, annual reports):
- EBITDA, profitability queries    → MD&A section (Management Discussion & Analysis)
- Liquidity, cash flow queries     → Cash Flow Statement + liquidity footnotes
- Risk factor queries              → Part I, Item 1A (Risk Factors)  
- Revenue breakdown queries        → Segment reporting or Item 7
- Forward-looking / strategy       → CEO letter, Outlook, Strategy section
- Debt, credit, leverage queries   → Balance Sheet + debt footnotes
- Regulatory / compliance queries  → Legal Proceedings or regulatory filings
"""

print("✅ Expert rules defined")
print("   These get injected into the retrieval prompt at query time.")



✅ Expert rules defined
   These get injected into the retrieval prompt at query time.


In [75]:
# ── Expert Routing Rules — Advanced Route of Learning AI ─────────────────────
FINANCIAL_EXPERT_RULES = """
Route queries to the correct module using these rules:
 
M1  Neural Network Refresher   → backprop, activations, optimizers, PyTorch basics
M2  Hardware                   → GPU, TPU, Apple Silicon, compute infrastructure
M3  Transformers 101           → attention, self-attention, encoder-decoder, MHA
M4  Tokenization               → BPE, WordPiece, SentencePiece, Byte Latent Transformers
M5  Finetuning Architectures   → hands-on BERT/GPT/T5 finetuning, Hugging Face
M6  KV Cache & Attention       → KV cache, Flash Attention, MQA, GQA, RoPE, vLLM
M7  Scaling Laws               → Kaplan, Chinchilla, compute-optimal training
M8  Mixture of Experts         → MoE, sparse computation, Mixture of Depths
M9  Modern LLM Finetuning      → LoRA, QLoRA, SFT, DPO, PPO, RLHF, GRPO, ORPO,
                                  quantization, TRL, Unsloth, synthetic data,
                                  reasoning models, evaluation, deployment
M10 SLM                        → small language models, pruning, when SLM vs LLM
M11 Knowledge Distillation     → student-teacher, soft labels, DistilBERT, DeepSeek-R1
M12 Hybrid Architectures       → Mamba, RWKV, SSMs, Jamba, Nemotron, beyond Transformers
M13 Vision Foundations         → ViT, patch embeddings, CLIP, SigLIP, DINOv2
M14 Visual Language Models     → VLM architecture, aligner, multimodal reasoning
M15 Stable Diffusion & DiT     → DDPM, latent diffusion, FLUX.1, ControlNet, DreamBooth
M16 Embedding Models           → dense, sparse, binary, Matryoshka, MRL, fine-tuning
M17 RAG                        → chunking, BM25, ColBERT, hybrid RAG, rerankers,
                                  self/corrective/adaptive/agentic RAG, Graph RAG,
                                  multi-modal RAG, ColPali, RAG security
M18 Context Engineering        → prompt vs context engineering, memory architecture,
                                  context compression, KV cache, agent context lifecycle
M19 DSPy                       → signatures, modules, MIPROv2, self-optimizing RAG
M20 Agents                     → ReAct, MCP, LangGraph, CrewAI, browser agents,
                                  A2A, guardrails, observability, evaluation
M21 RL                         → PPO, GRPO, DAPO, GSPO, CISPO, reward models,
                                  RLHF vs RLVR, policy gradient, DeepSeek-R1 training
 
Cross-cutting rules:
- "learning path / where to start"     → M1 → M2 → M3 in order
- "production / deployment / serving"  → M9 (quantization) + M20 (agents)
- "fine-tuning vs RAG"                 → M9 + M17 + M18
- "multimodal / vision + language"     → M13 + M14 + M17 (multi-modal RAG)
- "reasoning models / test-time RL"    → M9 (reasoning) + M21 (GRPO/DAPO)
"""

In [81]:
# Expert guided tree search 

def llm_tree_search_with_expert(
    query: str,
    tree: list,
    expert_rules: str,
    model: str = "openai/gpt-oss-20b"
) -> dict:
    """
    Same as llm_tree_search() but with domain expert rules injected.
    The LLM uses these rules to guide its reasoning.
    """
    
    def compress(nodes):
        out = []
        for n in nodes:
            entry = {"node_id": n["node_id"], "title": n["title"],
                     "page": n.get("page_index", "?"),
                     "summary": n.get("text", "")[:150]}
            if n.get("nodes"):
                entry["children"] = compress(n["nodes"])
            out.append(entry)
        return out

    prompt = f"""You are a domain expert analyzing a document.
Find all node IDs that most likely contain the answer to the query.
Use the expert routing rules below to guide your reasoning.

Query: {query}

Document Tree:
{json.dumps(compress(tree), indent=2)}

Expert Routing Rules (follow these carefully):
{expert_rules}

Reply ONLY in this JSON format:
{{
  "thinking": "<your reasoning, referencing the expert rules>",
  "node_list": ["node_id1", "node_id2"]
}}"""

    response = groq_client.chat.completions.create(
        model=model,
        messages=[{"role": "user", "content": prompt}],
        response_format={"type": "json_object"}
    )
    return json.loads(response.choices[0].message.content)

In [82]:
# ── Test expert-guided retrieval ─────────────────────────────────────────────
query = "Details of the modern llm finetuning?"

print(f"🔍 Query: {query}\n")

# Without expert rules
print("── Without Expert Rules ──")
basic   = llm_tree_search(query, pageindex_tree)
print("Nodes:", basic.get("node_list"))

print()

# With expert rules
print("── With Expert Rules ──")
guided  = llm_tree_search_with_expert(query, pageindex_tree, FINANCIAL_EXPERT_RULES)
print("Nodes:", guided.get("node_list"))
print("Reasoning:", guided.get("thinking", "")[:300])

🔍 Query: Details of the modern llm finetuning?

── Without Expert Rules ──
Nodes: ['0014', '0017']

── With Expert Rules ──
Nodes: []
Reasoning: The query is about "modern LLM finetuning", which falls under Expert Module M9 (Modern LLM Finetuning). The provided document is the original Transformer paper and contains sections on architecture, training, and results, but it does not cover modern finetuning techniques such as LoRA, QLoRA, SFT, D


In [83]:
# Full expert guided RAG 

def expert_rag(query: str, tree: list, rules: str) -> str:
    """Expert-guided end-to-end RAG pipeline."""
    result  = llm_tree_search_with_expert(query, tree, rules)
    nodes   = find_nodes_by_ids(tree, result.get("node_list", []))
    return generate_answer(query, nodes)

# Run it
answer = expert_rag(
    query="Details of the syllabus of modern llm finetuning",
    tree=pageindex_tree,
    rules=FINANCIAL_EXPERT_RULES
)
print(answer)

No details of a syllabus for modern LLM finetuning are provided in the supplied context (Section 5 Training, Page 7; Section 6 Results, Page 8).
